# FantasAI Injuries Ingestion

Pull injury report data from FantasAI Cloudflare Worker API (Sleeper data source) and land it in bronze and silver Delta tables.

In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

BASE_URL = "https://api.fantasai.net"

In [0]:
# Create bronze injuries table
spark.sql("""
CREATE TABLE IF NOT EXISTS main.fantasai.bronze_injuries (
  player_id STRING,
  player_name STRING,
  team STRING,
  position STRING,
  injury_status STRING,
  injury_body_part STRING,
  injury_notes STRING,
  ingested_at TIMESTAMP
)
USING DELTA
""")

# Create silver injuries table
spark.sql("""
CREATE TABLE IF NOT EXISTS main.fantasai.silver_injuries (
  player_id STRING,
  player_name STRING,
  team STRING,
  position STRING,
  injury_status STRING,
  injury_body_part STRING,
  injury_notes STRING,
  ingested_at TIMESTAMP
)
USING DELTA
""")

print("✓ Tables created")

In [0]:
# Fetch injuries from Cloudflare Worker API
response = requests.get(f"{BASE_URL}/api/v1/injuries", timeout=30)
response.raise_for_status()
response_data = response.json()

# Extract the injuries array from the nested response
payload = response_data.get("injuries", [])

print(f"Fetched {len(payload)} injury records")

# Convert to DataFrame rows
rows = []
for injury in payload:
    rows.append(
        Row(
            player_id=str(injury.get("player_id") or injury.get("id", "")),
            player_name=injury.get("player_name") or injury.get("full_name"),
            team=injury.get("team"),
            position=injury.get("position"),
            injury_status=injury.get("injury_status") or injury.get("status"),
            injury_body_part=injury.get("injury_body_part"),
            injury_notes=injury.get("injury_notes") or injury.get("description"),
        )
    )

injuries_df = spark.createDataFrame(rows)

display(injuries_df)

In [0]:
# Write to bronze table
bronze_df = injuries_df.withColumn("ingested_at", F.current_timestamp())

(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("main.fantasai.bronze_injuries")
)

print(f"✓ Wrote {bronze_df.count()} records to bronze_injuries")

In [0]:
# Transform and deduplicate for silver
silver_df = (
    bronze_df
    .select(
        F.col("player_id").cast("string"),
        F.col("player_name").cast("string"),
        F.col("team").cast("string"),
        F.col("position").cast("string"),
        F.col("injury_status").cast("string"),
        F.col("injury_body_part").cast("string"),
        F.col("injury_notes").cast("string"),
        F.col("ingested_at"),
    )
    .dropDuplicates(["player_id"])
)

display(silver_df)

In [0]:
# Write to silver table (overwrite with latest injury status)
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("main.fantasai.silver_injuries")
)

print(f"✓ Wrote {silver_df.count()} records to silver_injuries")